# 대림 M2 일반화 검증 — 600번대(보라)·700번대(검정) 24장
**업로드:** `daelim_colab_m2.zip` (600/700 사진 각 12장 + 구간별 카탈로그 + 파이프라인 + 파인튜닝 모델)

⚠️ 파인튜닝 모델은 신형 포맷이라 **paddlepaddle 3.3 이상** 필요 (셀1이 최신 GPU판 설치)
사용법: GPU(T4) 런타임 → 셀 순서대로. 셀1 후 **세션 다시 시작** 필수.

In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 최신 GPU paddle (>=3.3, 파인튜닝 모델 포맷 요구)
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr
import paddle; print('paddle', paddle.__version__)
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 패키지 업로드 + 배치
from google.colab import files
up = files.upload()   # daelim_colab_m2.zip
!unzip -oq daelim_colab_m2.zip -d work/
%cd work
!ls photos600/ | wc -l; ls photos700/ | wc -l

In [ ]:
# 3) 24장 실행 (GPU, 장당 ~1-2분) — 구간별 카탈로그로 각각 채점
import glob, subprocess, sys
JOBS = [('photos600', 'catalog_600.csv'), ('photos700', 'catalog_700.csv')]
for folder, catalog in JOBS:
    for p in sorted(glob.glob(f'{folder}/*.jpg')):
        print('='*30, p)
        r = subprocess.run([sys.executable, '-u', 'daelim_closeup.py', p,
                            '--rec_dir', 'korean_lowres_rec_infer', '--catalog', catalog],
                           capture_output=True, text=True)
        for ln in r.stdout.splitlines():
            if ln.startswith('['): print(ln)
        if r.returncode != 0: print(r.stderr[-1500:])

In [ ]:
# 4) 구간별 집계 — 프레임 투표 + 대출중 대조 (사진이 근접·광각 혼합이므로 합산=구간 커버리지)
import json, glob, csv
from collections import Counter, defaultdict
agg = {}
for folder, catalog in [('photos600', 'catalog_600.csv'), ('photos700', 'catalog_700.csv')]:
    stems = {p.split('/')[-1].rsplit('.', 1)[0] for p in glob.glob(f'{folder}/*.jpg')}
    votes = Counter(); mis = Counter(); per_frame = {}
    for f in sorted(glob.glob('out_ondevice/*_result.json')):
        stem = f.split('/')[-1].replace('closeup', '').replace('_ft_result.json', '')
        if stem not in stems: continue
        rows = json.load(open(f, encoding='utf-8'))
        per_frame[stem] = (sum(1 for r in rows if r['call']), len(rows))
        for r in rows:
            if r['call']:
                votes[r['call']] += 1
                if r.get('mis'): mis[r['call']] += 1
    cat_status = {}
    for r in csv.DictReader(open(catalog, encoding='utf-8-sig')):
        cat_status[r['call_number'].strip()] = r['status']
    loaned = [c for c in votes if cat_status.get(c) and cat_status[c] not in ('비치자료', '')]
    absent = [c for c in votes if c not in cat_status]
    name = folder.replace('photos', '') + '번대'
    print(f'===== {name} =====')
    for s, (m, t) in sorted(per_frame.items()): print(f'  {s}: {m}/{t}')
    print(f'  합산 고유 {len(votes)}권 (카탈로그 {len(cat_status)}권) · 2표 이상 {sum(1 for v in votes.values() if v >= 2)}권')
    print(f'  오배열 의심(투표): {dict(mis.most_common(8))}')
    print(f'  대출중/비비치인데 발견: {loaned}')
    print(f'  목록에 없는 청구기호: {absent}')
    agg[name] = {'votes': dict(votes), 'mis': dict(mis), 'loaned_on_shelf': loaned,
                 'absent': absent, 'per_frame': per_frame}
json.dump(agg, open('out_ondevice/aggregate_m2.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('저장: out_ondevice/aggregate_m2.json')

In [ ]:
# 5) 결과 다운로드 (AR 이미지 + JSON)
!zip -q -r ../daelim_m2_results.zip out_ondevice
from google.colab import files
files.download('../daelim_m2_results.zip')